# Clase 7 — Listas, diccionarios y datasets para IA

En la Clase 6 recorrimos un dataset con bucles y apareció algo importante: **Python representa los datos con estructuras**.  
Ese dataset de frutas no era mágico. Era una combinación de **listas** y **diccionarios**.

Hoy vamos a trabajar sobre esas estructuras porque son la base de casi todo lo que sigue:
- datasets pequeños para practicar,
- registros que llegan con errores,
- configuraciones de un modelo,
- y más adelante, incluso mensajes de chat con `role` y `content`.

**Objetivos de la clase:**
- Leer y modificar listas y diccionarios.
- Representar un dataset como `list[dict]`.
- Detectar problemas básicos de calidad del dato.
- Hacer una mini ETL en memoria para dejar un dataset listo para clasificar.


---
## 1. Listas: colecciones ordenadas

Una lista guarda varios elementos **en orden**.  
Podés acceder por posición, recortar una parte (`slicing`), agregar elementos con `append()` y recorrerla con un `for`.

En IA esto aparece todo el tiempo:
- una lista de pesos,
- una lista de registros,
- una lista de predicciones,
- una lista de mensajes de chat.


In [ ]:
# Una lista sencilla de nombres de frutas
frutas_observadas = ["manzana", "naranja", "limon", "uva"]

print("Primera fruta:", frutas_observadas[0])
print("Última fruta :", frutas_observadas[-1])
print("Sublista     :", frutas_observadas[1:3])

frutas_observadas.append("pera")
print("Lista actualizada:", frutas_observadas)

frutas_citricas = []
for fruta in frutas_observadas:
    if fruta in ["naranja", "limon"]:
        frutas_citricas.append(fruta)

print("Frutas cítricas:", frutas_citricas)


---
## 2. Diccionarios: un registro con nombre para cada dato

Un diccionario guarda pares **clave -> valor**.  
Es ideal cuando cada elemento tiene varias características con nombre propio.

Ejemplo mental:
- una fruta tiene `peso`,
- `color`,
- `dulzura`,
- `clase`.

Cuando el dato puede venir incompleto, `.get()` es muy útil porque evita errores y permite definir un valor por defecto.


In [ ]:
# Un registro individual representado como diccionario
registro = {
    "peso": " 182 ",
    "color": " Rojo ",
    "dulzura": 7,
}

print("Peso crudo       :", registro["peso"])
print("Clase por defecto:", registro.get("clase", "sin etiqueta"))

registro["clase"] = "manzana"
registro["color"] = registro["color"].strip().lower()

print("Registro actualizado:")
print(registro)


In [ ]:
# Un dataset simple: lista de diccionarios
dataset = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 170, "color": "verde",    "dulzura": 6, "clase": "manzana"},
    {"peso": 12,  "color": "morado",   "dulzura": 8, "clase": "uva"},
]

for fruta in dataset:
    print(
        f"{fruta['clase']:8s} | "
        f"peso={fruta['peso']:3d} | "
        f"color={fruta['color']}"
    )

clases_pesadas = []
for fruta in dataset:
    if fruta["peso"] > 150:
        clases_pesadas.append(fruta["clase"])

print("Clases con peso > 150g:", clases_pesadas)


---
## 3. Calidad del dato: antes de modelar hay que limpiar

Un modelo no aprende sobre "la realidad". Aprende sobre **lo que le damos**.

Por eso, antes de clasificar o predecir, conviene revisar problemas comunes:
- claves faltantes,
- números guardados como texto,
- espacios extra,
- mayúsculas/minúsculas inconsistentes,
- valores vacíos o imposibles.

Esta limpieza inicial ya es una forma simple de **ETL**:
- **Extract**: recibimos registros crudos,
- **Transform**: corregimos formato y normalizamos,
- **Load**: construimos un dataset limpio.


In [ ]:
# Limpiar registros simples en un solo bucle
registros_crudos = [
    {"peso": " 182 ", "color": " Rojo ",    "dulzura": "7", "clase": "Manzana"},
    {"peso": "143",   "color": "naranja ",  "dulzura": "6", "clase": "NARANJA"},
    {"peso": "  12",  "color": " Morado ",  "dulzura": "8", "clase": "Uva"},
]

dataset_limpio = []

for registro in registros_crudos:
    peso = int(str(registro.get("peso", "")).strip())
    color = str(registro.get("color", "")).strip().lower()
    dulzura = int(str(registro.get("dulzura", "")).strip())
    clase = str(registro.get("clase", "")).strip().lower()

    registro_limpio = {
        "peso": peso,
        "color": color,
        "dulzura": dulzura,
        "clase": clase,
    }
    dataset_limpio.append(registro_limpio)

print(dataset_limpio)


---
## 4. Mini ETL en memoria: separar válidos de inválidos

Además de limpiar, muchas veces necesitamos decidir si un registro **sirve** o no sirve para el análisis.  
En una ETL real, los registros inválidos pueden ir a una cola de revisión.

Acá vamos a hacer una versión chica:
- construir `dataset_limpio`,
- y separar `registros_invalidos`.


In [ ]:
# Separar datos válidos e inválidos
registros_crudos = [
    {"peso": "180", "color": " rojo ",   "dulzura": "7", "clase": "Manzana"},
    {"peso": "",    "color": "naranja",  "dulzura": "6", "clase": "Naranja"},
    {"peso": "95",  "color": None,       "dulzura": "3", "clase": "Limon"},
    {"peso": "14",  "color": "morado",   "dulzura": "8", "clase": "Uva"},
    {"peso": "165", "color": "verde",    "dulzura": "6", "clase": "Manzana"},
]

dataset_limpio = []
registros_invalidos = []

for registro in registros_crudos:
    peso_crudo = registro.get("peso")
    color_crudo = registro.get("color")
    dulzura_cruda = registro.get("dulzura")
    clase_cruda = registro.get("clase")

    peso = int(str(peso_crudo).strip()) if peso_crudo not in (None, "") else None
    color = str(color_crudo).strip().lower() if color_crudo not in (None, "") else ""
    dulzura = int(str(dulzura_cruda).strip()) if dulzura_cruda not in (None, "") else None
    clase = str(clase_cruda).strip().lower() if clase_cruda not in (None, "") else ""

    registro_limpio = {
        "peso": peso,
        "color": color,
        "dulzura": dulzura,
        "clase": clase,
    }

    if peso is None or color == "" or dulzura is None or clase == "":
        registros_invalidos.append(registro_limpio)
    else:
        dataset_limpio.append(registro_limpio)

print("Válidos  :", dataset_limpio)
print("Inválidos:", registros_invalidos)


---
## 5. Filtrar y contar: primeras preguntas sobre el dataset

Una vez que el dataset está limpio, ya podés hacer preguntas simples:
- ¿cuántas manzanas hay?
- ¿qué frutas tienen dulzura alta?
- ¿cuántas entradas válidas quedaron?

Estas operaciones parecen pequeñas, pero son la base de cualquier análisis posterior.


In [ ]:
# Contar clases y filtrar registros
dataset_limpio = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 170, "color": "verde",    "dulzura": 6, "clase": "manzana"},
    {"peso": 14,  "color": "morado",   "dulzura": 8, "clase": "uva"},
    {"peso": 95,  "color": "amarillo", "dulzura": 3, "clase": "limon"},
]

conteo_por_clase = {}
frutas_muy_dulces = []

for fruta in dataset_limpio:
    clase = fruta["clase"]
    conteo_por_clase[clase] = conteo_por_clase.get(clase, 0) + 1

    if fruta["dulzura"] >= 7:
        frutas_muy_dulces.append(fruta)

print("Conteo por clase:", conteo_por_clase)
print("Frutas muy dulces:", frutas_muy_dulces)


---
## 📝 Actividad 1 — Limpiar registros con faltantes e inconsistencias

En esta actividad vas a trabajar con un lote de registros crudos.

**Consigna:**
- Normalizá `peso`, `color`, `dulzura` y `clase`.
- Separá los válidos de los inválidos.
- Tratá como inválido cualquier registro que no tenga todos los campos clave.

**Pistas:**
- `.get()` para leer claves que podrían no estar.
- `str(...).strip().lower()` para texto.
- `int(...)` para números guardados como texto.


In [ ]:
# ACTIVIDAD 1: limpiar registros con faltantes e inconsistencias
registros_crudos = [
    {"peso": " 182 ", "color": " Rojo ",     "dulzura": "7", "clase": "Manzana"},
    {"peso": "",      "color": "naranja",    "dulzura": "6", "clase": "Naranja"},
    {"peso": " 14 ",  "color": " Morado ",   "dulzura": "8", "clase": "Uva"},
    {"peso": None,    "color": "verde",      "dulzura": "6", "clase": "Manzana"},
    {"peso": "95",    "color": None,         "dulzura": "3", "clase": "Limon"},
    {"peso": "130",   "color": " naranja ",  "dulzura": "5", "clase": "Naranja"},
]

dataset_limpio = []
registros_invalidos = []

for registro in registros_crudos:
    # TODO:
    # 1. Leer cada campo con .get()
    # 2. Normalizar peso/color/dulzura/clase
    # 3. Marcar como válido solo si no faltan datos clave
    registro_limpio = {
        "peso": None,
        "color": "",
        "dulzura": None,
        "clase": "",
    }

    es_valido = False

    if es_valido:
        dataset_limpio.append(registro_limpio)
    else:
        registros_invalidos.append(registro)

print("Válidos  :", dataset_limpio)
print("Inválidos:", registros_invalidos)

# Objetivo esperado:
# - colores en minúscula y sin espacios extra
# - pesos y dulzuras convertidos a números
# - registros incompletos separados como inválidos


---
### Actividad 2 — Filtrar y contar registros válidos

Ahora ya no hace falta limpiar. El dataset ya viene homogéneo.

**Consigna:**
- Contá cuántas frutas hay por clase.
- Guardá en otra lista las frutas con `dulzura >= 7`.
- Mostrá ambos resultados.


In [ ]:
# ACTIVIDAD 2: filtrar y contar registros válidos
dataset = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 170, "color": "verde",    "dulzura": 6, "clase": "manzana"},
    {"peso": 14,  "color": "morado",   "dulzura": 8, "clase": "uva"},
    {"peso": 95,  "color": "amarillo", "dulzura": 3, "clase": "limon"},
    {"peso": 130, "color": "naranja",  "dulzura": 5, "clase": "naranja"},
]

conteo_por_clase = {}
frutas_muy_dulces = []

for fruta in dataset:
    # TODO: actualizar conteo_por_clase
    # TODO: si dulzura >= 7, agregar a frutas_muy_dulces
    pass

print("Conteo por clase:", conteo_por_clase)
print("Frutas muy dulces:", frutas_muy_dulces)


---
### Actividad 3 — Transformar registros crudos a un formato homogéneo

A veces los datos llegan con otros nombres de campos.

**Consigna:**
Transformá cada registro de entrada al formato estándar:
```python
{"peso": ..., "color": ..., "dulzura": ..., "clase": ...}
```


In [ ]:
# ACTIVIDAD 3: transformar registros crudos a formato homogéneo
registros_origen = [
    {"peso_g": "180", "tono": "Rojo",     "azucar": 7, "etiqueta": "manzana"},
    {"peso_g": "140", "tono": "Naranja",  "azucar": 6, "etiqueta": "naranja"},
    {"peso_g": "12",  "tono": "Morado",   "azucar": 8, "etiqueta": "uva"},
]

dataset_homogeneo = []

for origen in registros_origen:
    nuevo_registro = {
        "peso": None,
        "color": "",
        "dulzura": None,
        "clase": "",
    }

    # TODO:
    # - copiar peso_g -> peso
    # - copiar tono -> color
    # - copiar azucar -> dulzura
    # - copiar etiqueta -> clase
    # - normalizar color y clase

    dataset_homogeneo.append(nuevo_registro)

print(dataset_homogeneo)


---
## ✅ Resumen

| Idea | Para qué sirve |
|---|---|
| Lista | Guardar muchos elementos ordenados |
| Diccionario | Guardar un registro con claves y valores |
| `list[dict]` | Representar un dataset simple |
| Limpieza / normalización | Corregir formatos antes de analizar |
| ETL básica | Convertir registros crudos en datos utilizables |

**Lo que construiste hoy:**
- datasets como listas de diccionarios,
- limpieza básica de registros,
- conteos y filtros simples sobre datos válidos.

**Lo que viene:**
- **Clase 8**: vamos a encapsular esta lógica en **funciones** para no repetir el mismo código cada vez.
- Las mismas operaciones de limpieza, validación y predicción van a pasar de ser bloques sueltos a piezas reutilizables.
